In [1]:
import sys
import subprocess

def pip(cmd):
    print(">>", " ".join(cmd))
    subprocess.check_call([sys.executable, "-m", "pip"] + cmd)

# ------------------------------------------------------------
# 0) Detect GPU without locking in old broken imports
# ------------------------------------------------------------
has_gpu = False
try:
    import torch
    has_gpu = torch.cuda.is_available()
except Exception:
    has_gpu = False

print("GPU detected:", has_gpu)

# ------------------------------------------------------------
# 1) Remove conflicting installs
# ------------------------------------------------------------
pip([
    "uninstall", "-y",
    "torch", "torchvision", "torchaudio",
    "torch-geometric",
    "pyg-lib", "torch-scatter", "torch-sparse",
    "torch-cluster", "torch-spline-conv",
    "numpy", "pandas", "captum"
])

# ------------------------------------------------------------
# 2) Install stable NumPy / Pandas for Colab
# ------------------------------------------------------------
pip(["install", "--no-cache-dir", "numpy==1.26.4", "pandas==2.2.2"])

# ------------------------------------------------------------
# 3) Install stable Torch
# ------------------------------------------------------------
TORCH_VER = "2.2.2"
TV_VER = "0.17.2"
TA_VER = "2.2.2"

if has_gpu:
    cu_tag = "cu121"
    pip([
        "install", "-q",
        f"torch=={TORCH_VER}",
        f"torchvision=={TV_VER}",
        f"torchaudio=={TA_VER}",
        "--index-url", "https://download.pytorch.org/whl/cu121"
    ])
else:
    cu_tag = "cpu"
    pip([
        "install", "-q",
        f"torch=={TORCH_VER}",
        f"torchvision=={TV_VER}",
        f"torchaudio=={TA_VER}",
        "--index-url", "https://download.pytorch.org/whl/cpu"
    ])

# ------------------------------------------------------------
# 4) Install PyTorch Geometric
# ------------------------------------------------------------
wheel_url = f"https://data.pyg.org/whl/torch-{TORCH_VER}+{cu_tag}.html"

pip(["install", "-q", "torch-geometric"])
pip([
    "install", "-q",
    "pyg-lib", "torch-scatter", "torch-sparse",
    "torch-cluster", "torch-spline-conv",
    "-f", wheel_url
])

# ------------------------------------------------------------
# 5) Install Captum
# ------------------------------------------------------------
pip(["install", "-q", "captum"])

print("\nInstallation finished successfully.")
print("IMPORTANT: now go to Runtime > Restart runtime")
print("After the restart, run your next notebook block.")

GPU detected: True
>> uninstall -y torch torchvision torchaudio torch-geometric pyg-lib torch-scatter torch-sparse torch-cluster torch-spline-conv numpy pandas captum
>> install --no-cache-dir numpy==1.26.4 pandas==2.2.2
>> install -q torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2 --index-url https://download.pytorch.org/whl/cu121
>> install -q torch-geometric
>> install -q pyg-lib torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-2.2.2+cu121.html
>> install -q captum

Installation finished successfully.
IMPORTANT: now go to Runtime > Restart runtime
After the restart, run your next notebook block.


In [1]:
# ----------------------------
# 0) MOUNT + IMPORTS
# ----------------------------
from google.colab import drive
drive.mount("/content/drive")

import os, math, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import average_precision_score

from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv


NEIGHBOR_OK = False
try:
    from torch_geometric.loader import NeighborLoader
    NEIGHBOR_OK = True
except Exception:
    NEIGHBOR_OK = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE, "| NeighborLoader available:", NEIGHBOR_OK)

# ----------------------------
# 1) LOAD ELLIPTIC DATA
# ----------------------------
BASE_DIR = "/content/drive/MyDrive/Thesis/Data sets"
FEATURES_PATH = os.path.join(BASE_DIR, "elliptic_txs_features.csv")
EDGES_PATH    = os.path.join(BASE_DIR, "elliptic_txs_edgelist.csv")
CLASSES_PATH  = os.path.join(BASE_DIR, "elliptic_txs_classes.csv")

assert os.path.exists(FEATURES_PATH), f"Missing: {FEATURES_PATH}"
assert os.path.exists(EDGES_PATH),    f"Missing: {EDGES_PATH}"
assert os.path.exists(CLASSES_PATH),  f"Missing: {CLASSES_PATH}"

feat_df = pd.read_csv(FEATURES_PATH, header=None).rename(columns={0:"txId", 1:"timestep"})
feat_df["txId"] = feat_df["txId"].astype(str)
feat_df["timestep"] = pd.to_numeric(feat_df["timestep"], errors="coerce").fillna(0).astype(int)

cls_df = pd.read_csv(CLASSES_PATH)
cls_df["txId"] = cls_df["txId"].astype(str)

edge_df = pd.read_csv(EDGES_PATH)
edge_df["txId1"] = edge_df["txId1"].astype(str)
edge_df["txId2"] = edge_df["txId2"].astype(str)

def map_label(v):
    if pd.isna(v): return -1
    s = str(v).strip().lower()
    if s == "unknown": return -1
    if s == "1": return 1   # illicit
    if s == "2": return 0   # licit
    return -1

cls_map = cls_df.set_index("txId")["class"].to_dict()
feat_df["y"] = feat_df["txId"].map(cls_map).apply(map_label).astype(int)

# Node index
tx_ids = feat_df["txId"].to_numpy()
tx2idx = {t:i for i,t in enumerate(tx_ids)}

# Filter edges to existing nodes
edge_df = edge_df[edge_df["txId1"].isin(tx2idx) & edge_df["txId2"].isin(tx2idx)].copy()
src = edge_df["txId1"].map(tx2idx).to_numpy(np.int64)
dst = edge_df["txId2"].map(tx2idx).to_numpy(np.int64)

# Undirected edge_index
src_u = np.concatenate([src, dst])
dst_u = np.concatenate([dst, src])
edge_index = torch.tensor(np.vstack([src_u, dst_u]), dtype=torch.long)

# Features matrix
exclude = {"txId", "timestep", "y"}
X_cols = [c for c in feat_df.columns if c not in exclude]
X_raw = feat_df[X_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy(np.float32)

y_np = feat_df["y"].to_numpy(np.int64)
t_np = feat_df["timestep"].to_numpy(np.int64)
TIMEPOINTS = np.sort(np.unique(t_np))
K = len(TIMEPOINTS)

print("Nodes:", len(tx_ids), "| Undirected edges:", edge_index.shape[1])
print("X:", X_raw.shape, "| K timepoints:", K, "| t range:", (int(TIMEPOINTS.min()), int(TIMEPOINTS.max())))
print("Known labels:", int((y_np != -1).sum()), "| Unknown:", int((y_np == -1).sum()))

# ----------------------------
# 2) TORCH BASE TENSORS (NO torch->numpy later)
# ----------------------------
x_all = torch.tensor(X_raw, dtype=torch.float32)  # [N,F]
y_all = torch.tensor(y_np, dtype=torch.long)      # [N]
t_all = torch.tensor(t_np, dtype=torch.long)      # [N]
edge_index_all = edge_index.long()                # [2,E]

# ----------------------------
# 3) TEMPORAL PROTOCOL (PDF-style)
# ----------------------------
L = 10       # window length in timesteps
s = 3        # retrain step
h = 3        # future eval slice length after B
VAL_FRAC_TP = 0.2


HP = dict(
    hidden=256,
    dropout=0.30,
    lr=5e-4,
    weight_decay=5e-4,
    epochs=80,
    patience=10,       # early stopping
)

R = 5  # seed replications

def window_timepoints(k_idx: int, L: int):
    start = max(0, k_idx - L + 1)
    return TIMEPOINTS[start:k_idx+1]

def time_aware_split(window_ts: np.ndarray, val_frac_tp: float):
    window_ts = np.array(window_ts, dtype=np.int64)
    if len(window_ts) <= 1:
        return window_ts, np.array([], dtype=np.int64)
    n_val = max(1, int(math.ceil(len(window_ts) * val_frac_tp)))
    val_ts = window_ts[-n_val:]
    train_ts = window_ts[:-n_val]
    if len(train_ts) == 0:
        train_ts, val_ts = window_ts[:-1], window_ts[-1:]
    return train_ts, val_ts

# ----------------------------
# 4) SUBGRAPH BUILDER
# ----------------------------
def build_subgraph_torch(cutoff_t: int):
    cutoff_t = int(cutoff_t)
    keep = (t_all <= cutoff_t)
    keep_idx = keep.nonzero(as_tuple=False).view(-1)  # old node indices kept

    # old -> new mapping
    map_old = torch.full((t_all.size(0),), -1, dtype=torch.long)
    map_old[keep_idx] = torch.arange(keep_idx.numel(), dtype=torch.long)

    src = edge_index_all[0]
    dst = edge_index_all[1]
    e_keep = keep[src] & keep[dst]

    src_k = src[e_keep]
    dst_k = dst[e_keep]

    src_new = map_old[src_k]
    dst_new = map_old[dst_k]
    edge_index_sub = torch.stack([src_new, dst_new], dim=0)

    data = Data(
        x=x_all[keep_idx].clone(),
        edge_index=edge_index_sub,
        y=y_all[keep_idx].clone()
    )
    data.t = t_all[keep_idx].clone()
    return data

# ----------------------------
# 5) NO-LEAKAGE SCALING (fit on training timepoints)
# ----------------------------
def fit_scaler(train_ts: np.ndarray):
    m = np.isin(t_np, train_ts)
    Xtr = X_raw[m]
    mu = Xtr.mean(axis=0, keepdims=True).astype(np.float32)
    sd = (Xtr.std(axis=0, keepdims=True) + 1e-6).astype(np.float32)
    return mu, sd

def apply_scaler(x: torch.Tensor, mu: np.ndarray, sd: np.ndarray):
    mu_t = torch.tensor(mu, dtype=torch.float32, device=x.device)
    sd_t = torch.tensor(sd, dtype=torch.float32, device=x.device)
    return (x - mu_t) / sd_t

# ----------------------------
# 6) MASK HELPERS
# ----------------------------
def labeled_mask(data: Data):
    return (data.y == 0) | (data.y == 1)

def mask_for_timepoints(data: Data, ts_np_array):
    ts_t = torch.tensor(ts_np_array, dtype=torch.long, device=data.t.device)
    return torch.isin(data.t, ts_t)

def build_masks_for_pair(data: Data, train_ts, val_ts, eval_ts):
    known = labeled_mask(data)
    train_mask = mask_for_timepoints(data, train_ts) & known
    val_mask   = mask_for_timepoints(data, val_ts)   & known
    eval_mask  = mask_for_timepoints(data, eval_ts)  & known
    return train_mask, val_mask, eval_mask

# ----------------------------
# 7) DRIFT SIGNALS (robust, no numpy-from-torch)
# ----------------------------
def total_variation_distance(p0, p1, q0, q1) -> float:
    return 0.5 * (abs(p0 - q0) + abs(p1 - q1))

def drift_X_meanShift(windowA_ts, windowB_ts) -> float:
    # mean absolute shift of per-feature means (proxy; fast and stable)
    mA = torch.isin(t_all, torch.tensor(windowA_ts, dtype=torch.long))
    mB = torch.isin(t_all, torch.tensor(windowB_ts, dtype=torch.long))
    XA = x_all[mA]
    XB = x_all[mB]
    if XA.numel() == 0 or XB.numel() == 0:
        return float("nan")
    return float(torch.mean(torch.abs(XA.mean(dim=0) - XB.mean(dim=0))).item())

def drift_Y_TV(windowA_ts, windowB_ts) -> float:
    tsA = torch.tensor(windowA_ts, dtype=torch.long)
    tsB = torch.tensor(windowB_ts, dtype=torch.long)
    mA = torch.isin(t_all, tsA) & ((y_all == 0) | (y_all == 1))
    mB = torch.isin(t_all, tsB) & ((y_all == 0) | (y_all == 1))
    yA = y_all[mA]
    yB = y_all[mB]
    if yA.numel() == 0 or yB.numel() == 0:
        return float("nan")
    p0 = float((yA == 0).float().mean().item())
    p1 = float((yA == 1).float().mean().item())
    q0 = float((yB == 0).float().mean().item())
    q1 = float((yB == 1).float().mean().item())
    return total_variation_distance(p0, p1, q0, q1)

# ----------------------------
# 8) GRAPHSAGE (3 layers) + TRAINING (class weights + early stopping)
# ----------------------------
class GraphSAGE3(nn.Module):
    def __init__(self, in_dim: int, hidden: int = 256, dropout: float = 0.3):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden)
        self.conv2 = SAGEConv(hidden, hidden)
        self.conv3 = SAGEConv(hidden, hidden)
        self.lin = nn.Linear(hidden, 2)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.conv3(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        return self.lin(x)

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

@torch.no_grad()
def pr_auc_on_mask(logits: torch.Tensor, y_true: torch.Tensor, mask: torch.Tensor) -> float:
    probs = F.softmax(logits[mask], dim=-1)[:, 1].detach().cpu().tolist()
    ys = y_true[mask].detach().cpu().tolist()
    return float(average_precision_score(ys, probs))

def compute_class_weights(labels: torch.Tensor) -> torch.Tensor:
    # labels are 0/1 only
    num_pos = int((labels == 1).sum().item())
    num_neg = int((labels == 0).sum().item())
    if num_pos == 0:
        # avoid div by 0; degenerate
        return torch.tensor([1.0, 1.0], device=labels.device)
    w_pos = num_neg / (num_pos + 1e-6)
    return torch.tensor([1.0, float(w_pos)], device=labels.device)

def train_fullbatch_one_seed(data: Data, train_mask: torch.Tensor, val_mask: torch.Tensor, seed: int, hp: dict):
    set_seed(seed)
    data = data.to(DEVICE)
    train_mask = train_mask.to(DEVICE)
    val_mask   = val_mask.to(DEVICE)

    model = GraphSAGE3(data.num_features, hidden=hp["hidden"], dropout=hp["dropout"]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])

    # (1) class weights
    w = compute_class_weights(data.y[train_mask])

    best_val = -1.0
    best_state = None
    bad_epochs = 0

    for ep in range(1, hp["epochs"] + 1):
        model.train()
        opt.zero_grad()
        logits = model(data.x, data.edge_index)

        loss = F.cross_entropy(logits[train_mask], data.y[train_mask], weight=w)
        loss.backward()
        opt.step()

        # Early stopping on val PR-AUC
        if val_mask.sum().item() > 0:
            model.eval()
            with torch.no_grad():
                logits = model(data.x, data.edge_index)
                val_pr = pr_auc_on_mask(logits, data.y, val_mask)

            if val_pr > best_val:
                best_val = val_pr
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                bad_epochs = 0
            else:
                bad_epochs += 1
                if bad_epochs >= hp["patience"]:
                    break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    return model, best_val

def train_neighbor_one_seed(data: Data, train_mask: torch.Tensor, val_mask: torch.Tensor, seed: int, hp: dict):
    # (5) Neighbor sampling, if it works; otherwise caller should fallback
    set_seed(seed)
    data = data.to(DEVICE)
    train_idx = train_mask.nonzero(as_tuple=False).view(-1).to(DEVICE)

    model = GraphSAGE3(data.num_features, hidden=hp["hidden"], dropout=hp["dropout"]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])

    # (1) class weights computed on train nodes
    w = compute_class_weights(data.y[train_mask.to(DEVICE)])

    # Try to construct loader; can fail if ops missing
    loader = NeighborLoader(
        data,
        input_nodes=train_idx,
        num_neighbors=[25, 15, 10],
        batch_size=2048,
        shuffle=True
    )

    best_val = -1.0
    best_state = None
    bad_epochs = 0

    for ep in range(1, hp["epochs"] + 1):
        model.train()
        for batch in loader:
            batch = batch.to(DEVICE)
            opt.zero_grad()
            out = model(batch.x, batch.edge_index)
            # first batch.batch_size are input nodes
            loss = F.cross_entropy(out[:batch.batch_size], batch.y[:batch.batch_size], weight=w)
            loss.backward()
            opt.step()

        if val_mask.sum().item() > 0:
            model.eval()
            with torch.no_grad():
                logits = model(data.x, data.edge_index)
                val_pr = pr_auc_on_mask(logits, data.y, val_mask.to(DEVICE))

            if val_pr > best_val:
                best_val = val_pr
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                bad_epochs = 0
            else:
                bad_epochs += 1
                if bad_epochs >= hp["patience"]:
                    break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    return model, best_val

def train_one_seed(data, train_mask, val_mask, seed, hp):
    # Use NeighborLoader if available AND actually works, else full-batch
    if NEIGHBOR_OK:
        try:
            return train_neighbor_one_seed(data, train_mask, val_mask, seed, hp)
        except Exception as e:
            print(f"[NeighborLoader failed -> fallback full-batch] seed={seed} err={type(e).__name__}: {e}")
    return train_fullbatch_one_seed(data, train_mask, val_mask, seed, hp)

# ----------------------------
# 9) RUN PDF-STYLE TEMPORAL PROTOCOL
# ----------------------------
def pr_auc_to_loss(pr_auc: float) -> float:
    return 1.0 - float(pr_auc)

records = []

max_k = K - (s + h) - 1
if max_k < (L - 1):
    raise ValueError("Not enough timepoints for chosen L/s/h. Reduce them.")

for k_idx in range(L - 1, max_k + 1):
    tA = int(TIMEPOINTS[k_idx])
    tB = int(TIMEPOINTS[k_idx + s])

    Wk   = window_timepoints(k_idx, L)
    Wk_s = window_timepoints(k_idx + s, L)

    trainA_ts, valA_ts = time_aware_split(Wk,   VAL_FRAC_TP)
    trainB_ts, valB_ts = time_aware_split(Wk_s, VAL_FRAC_TP)

    eval_ts = TIMEPOINTS[(k_idx + s + 1):(k_idx + s + 1 + h)]
    tEvalCut = int(np.max(eval_ts))

    # Drift context
    dX = drift_X_meanShift(Wk, Wk_s)
    dY = drift_Y_TV(Wk, Wk_s)

    # Training graphs
    dataA = build_subgraph_torch(tA)
    dataB = build_subgraph_torch(tB)

    # No-leakage scaling
    muA, sdA = fit_scaler(trainA_ts)
    muB, sdB = fit_scaler(trainB_ts)

    dataA.x = apply_scaler(dataA.x.to(DEVICE), muA, sdA).cpu()
    dataB.x = apply_scaler(dataB.x.to(DEVICE), muB, sdB).cpu()

    train_mask_A, val_mask_A, _ = build_masks_for_pair(dataA, trainA_ts, valA_ts, eval_ts)
    train_mask_B, val_mask_B, _ = build_masks_for_pair(dataB, trainB_ts, valB_ts, eval_ts)

    # Eval graph must include future slice
    dataE = build_subgraph_torch(tEvalCut)

    # Scaled eval graphs (each model gets its own scaler)
    dataE_A = Data(x=dataE.x.clone(), edge_index=dataE.edge_index, y=dataE.y.clone()); dataE_A.t = dataE.t.clone()
    dataE_B = Data(x=dataE.x.clone(), edge_index=dataE.edge_index, y=dataE.y.clone()); dataE_B.t = dataE.t.clone()

    dataE_A.x = apply_scaler(dataE_A.x.to(DEVICE), muA, sdA)
    dataE_B.x = apply_scaler(dataE_B.x.to(DEVICE), muB, sdB)

    knownE = ((dataE.y == 0) | (dataE.y == 1))
    eval_mask_E = torch.isin(dataE.t, torch.tensor(eval_ts, dtype=torch.long)) & knownE
    if eval_mask_E.sum().item() < 50:
        continue

    # Fraude rate evaluatieset
    y_eval = dataE.y[eval_mask_E]
    fraud_rate_eval = float((y_eval == 1).float().mean().item())

    # Fraude rate window A (enkel gekende labels)
    mask_WA = torch.isin(t_all, torch.tensor(Wk, dtype=torch.long)) & ((y_all == 0) | (y_all == 1))
    y_WA = y_all[mask_WA]
    fraud_rate_A_window = float((y_WA == 1).float().mean().item()) if y_WA.numel() > 0 else float("nan")

    # Fraude rate window B (enkel gekende labels)
    mask_WB = torch.isin(t_all, torch.tensor(Wk_s, dtype=torch.long)) & ((y_all == 0) | (y_all == 1))
    y_WB = y_all[mask_WB]
    fraud_rate_B_window = float((y_WB == 1).float().mean().item()) if y_WB.numel() > 0 else float("nan")

    prA_list, prB_list, vA_list, vB_list = [], [], [], []

    for r in range(R):
        seedA = 100000 + k_idx * 100 + r
        seedB = 200000 + k_idx * 100 + r

        modelA, vA = train_one_seed(dataA, train_mask_A, val_mask_A, seedA, HP)
        modelB, vB = train_one_seed(dataB, train_mask_B, val_mask_B, seedB, HP)
        vA_list.append(vA); vB_list.append(vB)

        with torch.no_grad():
            logitsA = modelA.to(DEVICE)(dataE_A.x, dataE_A.edge_index.to(DEVICE))
            logitsB = modelB.to(DEVICE)(dataE_B.x, dataE_B.edge_index.to(DEVICE))

            prA = pr_auc_on_mask(logitsA, dataE_A.y.to(DEVICE), eval_mask_E.to(DEVICE))
            prB = pr_auc_on_mask(logitsB, dataE_B.y.to(DEVICE), eval_mask_E.to(DEVICE))

        prA_list.append(prA); prB_list.append(prB)

    prA_mean = float(sum(prA_list) / len(prA_list))
    prB_mean = float(sum(prB_list) / len(prB_list))

    lossA = pr_auc_to_loss(prA_mean)
    lossB = pr_auc_to_loss(prB_mean)
    delta_perf = float(lossA - lossB)

    records.append({
        "k_idx": int(k_idx),
        "tA": tA, "tB": tB,
        "eval_start": int(eval_ts[0]), "eval_end": int(eval_ts[-1]),
        "PR_A_mean": prA_mean,
        "PR_B_mean": prB_mean,
        "delta_perf_loss(A-B)": delta_perf,
        "valA_mean": float(np.nanmean(vA_list)),
        "valB_mean": float(np.nanmean(vB_list)),
        "PR_A_std": float(np.std(prA_list)),
        "PR_B_std": float(np.std(prB_list)),
        "delta_X_meanShift": float(dX),
        "delta_Y_TV": float(dY),
        "n_eval": int(eval_mask_E.sum().item()),
        "fraud_rate_eval": fraud_rate_eval,
        "fraud_rate_A_window": fraud_rate_A_window,
        "fraud_rate_B_window": fraud_rate_B_window,
    })

results_df = pd.DataFrame(records)
print("Pairs computed:", len(results_df))
display(results_df.head(34))
display(results_df.sort_values("delta_perf_loss(A-B)", ascending=False).head(10))
display(results_df[[
    "PR_A_mean","PR_B_mean","delta_perf_loss(A-B)",
    "delta_X_meanShift","delta_Y_TV",
    "fraud_rate_eval","fraud_rate_A_window","fraud_rate_B_window"
]].describe())

Mounted at /content/drive
DEVICE: cuda | NeighborLoader available: True
Nodes: 203769 | Undirected edges: 468710
X: (203769, 165) | K timepoints: 49 | t range: (1, 49)
Known labels: 46564 | Unknown: 157205
Pairs computed: 34


,k_idx,tA,tB,eval_start,eval_end,PR_A_mean,PR_B_mean,delta_perf_loss(A-B),valA_mean,valB_mean,PR_A_std,PR_B_std,delta_X_meanShift,delta_Y_TV,n_eval,fraud_rate_eval
0,9,10,13,14,16,0.925873,0.963789,0.037915,0.913951,0.970648,0.020518,0.003020,0.035774,0.050162,1565,0.203195
1,10,11,14,15,17,0.947296,0.958900,0.011604,0.929907,0.969577,0.004246,0.003752,0.033468,0.046319,1959,0.190914
2,11,12,15,16,18,0.875484,0.943497,0.068013,0.967724,0.978689,0.011425,0.006533,0.043242,0.078507,1730,0.161272
3,12,13,16,17,19,0.797885,0.874232,0.076347,0.974033,0.988389,0.024039,0.019611,0.043061,0.062606,1945,0.118766
4,13,14,17,18,20,0.812823,0.852153,0.039331,0.970307,0.968476,0.014955,0.012923,0.048861,0.058465,2034,0.192724
5,14,15,18,19,21,0.827700,0.833475,0.005774,0.977706,0.901269,0.009400,0.003830,0.038527,0.040116,2286,0.192476
6,15,16,19,20,22,0.762945,0.846560,0.083615,0.988767,0.896475,0.039663,0.005148,0.038635,0.000014,3304,0.156780
7,16,17,20,21,23,0.649142,0.791799,0.142657,0.966880,0.889056,0.017891,0.010270,0.035960,0.031511,3591,0.086605
8,17,18,21,22,24,0.556395,0.618224,0.061828,0.891364,0.873180,0.023382,0.004578,0.034500,0.011272,4076,0.085378
9,18,19,22,23,25,0.600285,0.626022,0.025736,0.901414,0.889641,0.006019,0.004319,0.030945,0.023363,2907,0.105951


,k_idx,tA,tB,eval_start,eval_end,PR_A_mean,PR_B_mean,delta_perf_loss(A-B),valA_mean,valB_mean,PR_A_std,PR_B_std,delta_X_meanShift,delta_Y_TV,n_eval,fraud_rate_eval
25,34,35,38,39,41,0.401928,0.759897,0.357969,0.794281,0.893863,0.068694,0.013758,0.031945,0.042738,3526,0.087635
24,33,34,37,38,40,0.559452,0.785189,0.225737,0.787449,0.663253,0.039164,0.016841,0.036659,0.049756,3150,0.096508
13,22,23,26,27,29,0.720310,0.940862,0.220551,0.861792,0.956971,0.011119,0.013677,0.030961,0.007041,1664,0.263221
14,23,24,27,28,30,0.736249,0.939948,0.203699,0.428601,0.902850,0.006517,0.039474,0.023335,0.005766,1982,0.250757
7,16,17,20,21,23,0.649142,0.791799,0.142657,0.966880,0.889056,0.017891,0.010270,0.035960,0.031511,3591,0.086605
11,20,21,24,25,27,0.808943,0.917652,0.108708,0.873907,0.429543,0.015013,0.006897,0.035846,0.051635,1317,0.180714
6,15,16,19,20,22,0.762945,0.846560,0.083615,0.988767,0.896475,0.039663,0.005148,0.038635,0.000014,3304,0.156780
19,28,29,32,33,35,0.783778,0.866063,0.082285,0.970206,0.947533,0.023243,0.012977,0.034774,0.017535,2297,0.105355
21,30,31,34,35,37,0.612393,0.692786,0.080393,0.919829,0.794536,0.056423,0.056225,0.063397,0.050616,3547,0.071892
3,12,13,16,17,19,0.797885,0.874232,0.076347,0.974033,0.988389,0.024039,0.019611,0.043061,0.062606,1945,0.118766


KeyError: "['fraud_pct_eval'] not in index"

Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [2]:
import math
import random
import itertools
import warnings
from typing import List

from captum.attr import IntegratedGradients

from scipy.stats import spearmanr

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 0) CONFIG
# ------------------------------------------------------------
B0_SIZE = 100                 # fixed raw background size (used to derive one fixed IG baseline)
IG_N_STEPS = 32               # number of IG integration steps
IG_METHOD = "gausslegendre"   # Captum supports gausslegendre / riemann_* variants
MAX_EVAL_EXPLAIN = 64         # max eval nodes explained per pair
EPS = 1e-8
BASE_SEED = 12345

feature_names = list(X_cols)

# ------------------------------------------------------------
# 1) FIXED RAW BACKGROUND B0 (same across all timepoints)
#    We derive ONE fixed raw baseline vector from this background.
# ------------------------------------------------------------
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_all_seeds(BASE_SEED)

def make_fixed_background_B0(n_background=100, seed=12345):
    rng = np.random.default_rng(seed)
    n = X_raw.shape[0]
    take = min(n_background, n)
    idx = rng.choice(n, size=take, replace=False)
    return X_raw[idx].astype(np.float32)

B0_raw = make_fixed_background_B0(n_background=B0_SIZE, seed=BASE_SEED)

# Single fixed raw baseline for IG:
# Mean of the fixed raw background
IG_BASELINE_RAW = B0_raw.mean(axis=0, keepdims=True).astype(np.float32)

print("B0_raw shape:", B0_raw.shape)
print("IG_BASELINE_RAW shape:", IG_BASELINE_RAW.shape)

# ------------------------------------------------------------
# 2) HELPERS
# ------------------------------------------------------------
def normalize_attr_rows(mat: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    """
    Row-wise normalize by L1 magnitude:
        a_norm = a / (sum_j |a_j| + eps)
    """
    denom = np.sum(np.abs(mat), axis=1, keepdims=True) + eps
    return mat / denom

def dloc_matrix(phi1: np.ndarray, phi2: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    """
    Local distance per instance after row-wise L1 normalization.
    phi1, phi2: [N, F]
    returns: [N]
    """
    a = normalize_attr_rows(phi1, eps=eps)
    b = normalize_attr_rows(phi2, eps=eps)
    return np.sum(np.abs(a - b), axis=1)

def compute_delta_eloc(phiA_mean: np.ndarray, phiB_mean: np.ndarray, eps: float = 1e-8) -> float:
    """
    ΔEloc(A,B) = median_x dloc(phiA_bar(x), phiB_bar(x))
    """
    d = dloc_matrix(phiA_mean, phiB_mean, eps=eps)
    return float(np.median(d))

def compute_global_importance(phi_mean: np.ndarray) -> np.ndarray:
    """
    g(j) = mean_x |phi_j(x)|
    phi_mean: [N, F]
    returns: [F]
    """
    return np.mean(np.abs(phi_mean), axis=0)

def compute_delta_eglob(gA: np.ndarray, gB: np.ndarray) -> float:
    """
    ΔEglob(A,B) = 1 - Spearman(gA, gB)
    """
    rho, _ = spearmanr(gA, gB)
    if rho is None or np.isnan(rho):
        rho = 0.0
    return float(1.0 - rho)

def compute_topk_overlap(gA: np.ndarray, gB: np.ndarray, k: int = 10) -> float:
    idxA = np.argsort(-gA)[:k]
    idxB = np.argsort(-gB)[:k]
    return float(len(set(idxA).intersection(set(idxB))) / k)

def compute_sigma_from_seed_explanations(phi_seed_stack: np.ndarray, eps: float = 1e-8) -> float:
    """
    phi_seed_stack: [R, N, F]
    σE = median_x median_{r<r'} dloc(phi_r(x), phi_r'(x))
    """
    R_here, N_here, F_here = phi_seed_stack.shape
    pairwise_per_node = []
    for i in range(N_here):
        dists = []
        for r1 in range(R_here):
            for r2 in range(r1 + 1, R_here):
                a = phi_seed_stack[r1, i:i+1, :]
                b = phi_seed_stack[r2, i:i+1, :]
                d = dloc_matrix(a, b, eps=eps)[0]
                dists.append(float(d))
        pairwise_per_node.append(np.median(dists))
    return float(np.median(pairwise_per_node))

def choose_eval_node_indices(eval_mask_E: torch.Tensor, max_eval_explain=None, seed=12345):
    idx = eval_mask_E.nonzero(as_tuple=False).view(-1).cpu().tolist()
    if max_eval_explain is None or len(idx) <= max_eval_explain:
        return idx
    rng = np.random.default_rng(seed)
    chosen = rng.choice(np.array(idx), size=max_eval_explain, replace=False)
    chosen = np.sort(chosen).tolist()
    return chosen

def top10_long_df(g: np.ndarray, pair_id: int, model_name: str, explainer_name: str) -> pd.DataFrame:
    order = np.argsort(-g)[:10]
    rows = []
    for rank, j in enumerate(order, start=1):
        rows.append({
            "explainer": explainer_name,
            "pair_id": int(pair_id),
            "model": model_name,
            "rank": int(rank),
            "feature": feature_names[j],
            "importance": float(g[j]),
        })
    return pd.DataFrame(rows)

def top10_wide_row(g: np.ndarray, pair_id: int, model_name: str, explainer_name: str) -> dict:
    order = np.argsort(-g)[:10]
    row = {
        "explainer": explainer_name,
        "pair_id": int(pair_id),
        "model": model_name,
    }
    for rank, j in enumerate(order, start=1):
        row[f"f{rank}"] = feature_names[j]
        row[f"imp{rank}"] = float(g[j])
    return row

# ------------------------------------------------------------
# 3) MODEL WRAPPER FOR IG:
#    forward function takes ONLY target-node features as input,
#    while graph structure + neighbor features stay fixed.
# ------------------------------------------------------------
class TargetNodeIGWrapper(torch.nn.Module):
    """
    Wraps the GraphSAGE model so Integrated Gradients can attribute over a
    single target node's feature vector only, while keeping all other node
    features and graph structure fixed.
    """
    def __init__(self, model, dataE_scaled: Data, target_node_idx: int):
        super().__init__()
        self.model = model.to(DEVICE)
        self.model.eval()

        self.target_node_idx = int(target_node_idx)
        self.register_buffer("edge_index_buf", dataE_scaled.edge_index.clone().to(DEVICE))
        self.register_buffer("base_x_buf", dataE_scaled.x.clone().to(DEVICE))

    def forward(self, target_x_scaled: torch.Tensor) -> torch.Tensor:
        """
        target_x_scaled: [B, F]
        returns: p(class=1) for the target node, shape [B]
        """
        if target_x_scaled.dim() == 1:
            target_x_scaled = target_x_scaled.unsqueeze(0)

        outs = []
        for b in range(target_x_scaled.size(0)):
            x_mod = self.base_x_buf.clone()
            x_mod[self.target_node_idx] = target_x_scaled[b]
            logits = self.model(x_mod, self.edge_index_buf)
            p1 = F.softmax(logits[self.target_node_idx], dim=-1)[1]
            outs.append(p1)

        return torch.stack(outs, dim=0)

# ------------------------------------------------------------
# 4) IG explanation for one model on selected eval nodes
# ------------------------------------------------------------
def explain_eval_nodes_ig(
    model,
    dataE_scaled: Data,
    eval_node_indices: List[int],
    mu: np.ndarray,
    sd: np.ndarray,
    ig_baseline_raw: np.ndarray,
    ig_n_steps: int = 32,
    ig_method: str = "gausslegendre",
) -> np.ndarray:
    """
    Returns IG explanations with shape [N_eval_used, F]

    Important:
    - IG runs in the SCALED feature space actually seen by the model.
    - We start from a FIXED RAW baseline vector, then scale it with the model's
      own (mu, sd), so the baseline is comparable in raw space across time but
      valid in the model input space.
    """
    phi_all = []

    mu_1d = mu.reshape(-1).astype(np.float32)
    sd_1d = sd.reshape(-1).astype(np.float32)
    baseline_scaled = ((ig_baseline_raw.reshape(-1).astype(np.float32) - mu_1d) / sd_1d)

    baseline_t = torch.tensor(baseline_scaled, dtype=torch.float32, device=DEVICE).unsqueeze(0)

    for node_idx in eval_node_indices:
        wrapper = TargetNodeIGWrapper(
            model=model,
            dataE_scaled=dataE_scaled,
            target_node_idx=node_idx,
        )

        ig = IntegratedGradients(wrapper)

        x_scaled_target = dataE_scaled.x[node_idx].detach().clone().to(DEVICE).unsqueeze(0)
        x_scaled_target.requires_grad_(True)

        attr = ig.attribute(
            inputs=x_scaled_target,
            baselines=baseline_t,
            target=None,            # wrapper already outputs scalar p(class=1)
            n_steps=ig_n_steps,
            method=ig_method,
            return_convergence_delta=False,
        )

        phi_all.append(attr.detach().cpu().numpy()[0].astype(np.float64))

    return np.asarray(phi_all, dtype=np.float64)

# ------------------------------------------------------------
# 5) TEMPORAL LOOP — Integrated Gradients
# ------------------------------------------------------------
ig_summary_rows = []
ig_top10_long_parts = []
ig_top10_wide_rows = []

pair_counter = 0

#L=10, s=3, h=3
for k_idx in range(L - 1, K - s - h):
    Wk   = window_timepoints(k_idx, L)
    Wk_s = window_timepoints(k_idx + s, L)

    if len(Wk) < L or len(Wk_s) < L:
        continue

    trainA_ts, valA_ts = time_aware_split(Wk, VAL_FRAC_TP)
    trainB_ts, valB_ts = time_aware_split(Wk_s, VAL_FRAC_TP)

    tA = int(Wk[-1])
    tB = int(Wk_s[-1])

    eval_ts = TIMEPOINTS[k_idx + s + 1 : k_idx + s + 1 + h]
    if len(eval_ts) < h:
        continue

    tEvalCut = int(np.max(eval_ts))

    # training graphs
    dataA = build_subgraph_torch(tA)
    dataB = build_subgraph_torch(tB)

    # time-aware no-leakage scaling
    muA, sdA = fit_scaler(trainA_ts)
    muB, sdB = fit_scaler(trainB_ts)

    dataA.x = apply_scaler(dataA.x.to(DEVICE), muA, sdA).cpu()
    dataB.x = apply_scaler(dataB.x.to(DEVICE), muB, sdB).cpu()

    train_mask_A, val_mask_A, _ = build_masks_for_pair(dataA, trainA_ts, valA_ts, eval_ts)
    train_mask_B, val_mask_B, _ = build_masks_for_pair(dataB, trainB_ts, valB_ts, eval_ts)

    # common future eval graph
    dataE = build_subgraph_torch(tEvalCut)
    dataE_A = Data(x=dataE.x.clone(), edge_index=dataE.edge_index, y=dataE.y.clone())
    dataE_A.t = dataE.t.clone()

    dataE_B = Data(x=dataE.x.clone(), edge_index=dataE.edge_index, y=dataE.y.clone())
    dataE_B.t = dataE.t.clone()

    dataE_A.x = apply_scaler(dataE_A.x.to(DEVICE), muA, sdA).cpu()
    dataE_B.x = apply_scaler(dataE_B.x.to(DEVICE), muB, sdB).cpu()

    knownE = ((dataE.y == 0) | (dataE.y == 1))
    eval_mask_E = torch.isin(dataE.t, torch.tensor(eval_ts, dtype=torch.long)) & knownE

    if eval_mask_E.sum().item() < 20:
        continue

    eval_node_indices = choose_eval_node_indices(
        eval_mask_E=eval_mask_E,
        max_eval_explain=MAX_EVAL_EXPLAIN,
        seed=BASE_SEED + k_idx
    )

    if len(eval_node_indices) == 0:
        continue

    # train R seed models and explain selected eval nodes
    phiA_seeds = []
    phiB_seeds = []

    for r in range(R):
        seedA = 100000 + k_idx * 100 + r
        seedB = 200000 + k_idx * 100 + r

        modelA, valA = train_one_seed(dataA, train_mask_A, val_mask_A, seedA, HP)
        modelB, valB = train_one_seed(dataB, train_mask_B, val_mask_B, seedB, HP)

        phiA_r = explain_eval_nodes_ig(
            model=modelA,
            dataE_scaled=dataE_A,
            eval_node_indices=eval_node_indices,
            mu=muA,
            sd=sdA,
            ig_baseline_raw=IG_BASELINE_RAW,
            ig_n_steps=IG_N_STEPS,
            ig_method=IG_METHOD,
        )

        phiB_r = explain_eval_nodes_ig(
            model=modelB,
            dataE_scaled=dataE_B,
            eval_node_indices=eval_node_indices,
            mu=muB,
            sd=sdB,
            ig_baseline_raw=IG_BASELINE_RAW,
            ig_n_steps=IG_N_STEPS,
            ig_method=IG_METHOD,
        )

        phiA_seeds.append(phiA_r)
        phiB_seeds.append(phiB_r)

    phiA_seeds = np.stack(phiA_seeds, axis=0)   # [R, N_eval, F]
    phiB_seeds = np.stack(phiB_seeds, axis=0)

    phiA_mean = phiA_seeds.mean(axis=0)         # [N_eval, F]
    phiB_mean = phiB_seeds.mean(axis=0)


    delta_eloc = compute_delta_eloc(phiA_mean, phiB_mean, eps=EPS)
    gA = compute_global_importance(phiA_mean)
    gB = compute_global_importance(phiB_mean)
    delta_eglob = compute_delta_eglob(gA, gB)
    sigma_A = compute_sigma_from_seed_explanations(phiA_seeds, eps=EPS)
    sigma_B = compute_sigma_from_seed_explanations(phiB_seeds, eps=EPS)
    drift_ratio = float(delta_eloc / (sigma_A + EPS))
    top10_overlap = compute_topk_overlap(gA, gB, k=10)

    ig_summary_rows.append({
        "pair_id": int(pair_counter),
        "k_idx": int(k_idx),
        "tA": int(tA),
        "tB": int(tB),
        "eval_start": int(eval_ts[0]),
        "eval_end": int(eval_ts[-1]),
        "n_eval_total": int(eval_mask_E.sum().item()),
        "n_eval_used": int(len(eval_node_indices)),
        "ΔEloc_IG": float(delta_eloc),
        "ΔEglob_IG": float(delta_eglob),
        "σE(A)_IG": float(sigma_A),
        "σE(B)_IG": float(sigma_B),
        "DriftRatio_IG": float(drift_ratio),
        "Top10Overlap_IG": float(top10_overlap),
    })

    ig_top10_long_parts.append(top10_long_df(gA, pair_counter, "A", "IG"))
    ig_top10_long_parts.append(top10_long_df(gB, pair_counter, "B", "IG"))
    ig_top10_wide_rows.append(top10_wide_row(gA, pair_counter, "A", "IG"))
    ig_top10_wide_rows.append(top10_wide_row(gB, pair_counter, "B", "IG"))

    print(f"[IG] done pair_id={pair_counter} | k_idx={k_idx} | tA={tA} | tB={tB} | n_eval_used={len(eval_node_indices)}")
    pair_counter += 1

# ------------------------------------------------------------
# 6) OUTPUT TABLES — IG
# ------------------------------------------------------------
results_ig_df = pd.DataFrame(ig_summary_rows)
top10_ig_df = pd.concat(ig_top10_long_parts, ignore_index=True) if len(ig_top10_long_parts) else pd.DataFrame()
top10_ig_wide_df = pd.DataFrame(ig_top10_wide_rows)

if len(results_ig_df):
    results_ig_df = results_ig_df.sort_values(["tB", "pair_id"]).reset_index(drop=True)

print("\n================ IG SUMMARY TABLE ================\n")
display(results_ig_df)

print("\n================ IG TOP-10 (LONG) ================\n")
display(top10_ig_df)

print("\n================ IG TOP-10 (WIDE) ================\n")
display(top10_ig_wide_df)

# most interesting pairs first
if len(results_ig_df):
    print("\n================ IG sorted by DriftRatio ================\n")
    display(results_ig_df.sort_values("DriftRatio_IG", ascending=False).reset_index(drop=True))

B0_raw shape: (100, 165)
IG_BASELINE_RAW shape: (1, 165)
[IG] done pair_id=0 | k_idx=9 | tA=10 | tB=13 | n_eval_used=64
[IG] done pair_id=1 | k_idx=10 | tA=11 | tB=14 | n_eval_used=64
[IG] done pair_id=2 | k_idx=11 | tA=12 | tB=15 | n_eval_used=64
[IG] done pair_id=3 | k_idx=12 | tA=13 | tB=16 | n_eval_used=64
[IG] done pair_id=4 | k_idx=13 | tA=14 | tB=17 | n_eval_used=64
[IG] done pair_id=5 | k_idx=14 | tA=15 | tB=18 | n_eval_used=64
[IG] done pair_id=6 | k_idx=15 | tA=16 | tB=19 | n_eval_used=64
[IG] done pair_id=7 | k_idx=16 | tA=17 | tB=20 | n_eval_used=64
[IG] done pair_id=8 | k_idx=17 | tA=18 | tB=21 | n_eval_used=64
[IG] done pair_id=9 | k_idx=18 | tA=19 | tB=22 | n_eval_used=64
[IG] done pair_id=10 | k_idx=19 | tA=20 | tB=23 | n_eval_used=64
[IG] done pair_id=11 | k_idx=20 | tA=21 | tB=24 | n_eval_used=64
[IG] done pair_id=12 | k_idx=21 | tA=22 | tB=25 | n_eval_used=64
[IG] done pair_id=13 | k_idx=22 | tA=23 | tB=26 | n_eval_used=64
[IG] done pair_id=14 | k_idx=23 | tA=24 | tB

,pair_id,k_idx,tA,tB,eval_start,eval_end,n_eval_total,n_eval_used,ΔEloc_IG,ΔEglob_IG,σE(A)_IG,σE(B)_IG,DriftRatio_IG,Top10Overlap_IG
0,0,9,10,13,14,16,1565,64,0.822578,0.236129,0.735208,0.510062,1.118837,0.7
1,1,10,11,14,15,17,1959,64,0.626725,0.169094,0.580925,0.500238,1.078839,0.9
2,2,11,12,15,16,18,1730,64,0.640325,0.183282,0.505316,0.547659,1.267177,0.5
3,3,12,13,16,17,19,1945,64,0.659388,0.290642,0.476652,0.587792,1.383372,0.6
4,4,13,14,17,18,20,2034,64,0.647722,0.175669,0.516545,0.557318,1.253949,0.8
5,5,14,15,18,19,21,2286,64,0.535365,0.159108,0.577933,0.519013,0.926345,0.8
6,6,15,16,19,20,22,3304,64,0.593034,0.130812,0.581385,0.515832,1.020037,0.8
7,7,16,17,20,21,23,3591,64,0.554247,0.143862,0.517729,0.555072,1.070534,0.7
8,8,17,18,21,22,24,4076,64,0.678493,0.233620,0.548264,0.544566,1.237530,0.7
9,9,18,19,22,23,25,2907,64,0.635070,0.279678,0.493100,0.590799,1.287915,0.6



================ IG TOP-10 (LONG) ================



,explainer,pair_id,model,rank,feature,importance
0,IG,0,A,1,4,0.033501
1,IG,0,A,2,151,0.015382
2,IG,0,A,3,143,0.014524
3,IG,0,A,4,102,0.011796
4,IG,0,A,5,54,0.011544
...,...,...,...,...,...,...
675,IG,33,B,6,137,0.007272
676,IG,33,B,7,101,0.005580
677,IG,33,B,8,54,0.005336
678,IG,33,B,9,91,0.004682



================ IG TOP-10 (WIDE) ================



,explainer,pair_id,model,f1,imp1,f2,imp2,f3,imp3,f4,...,f6,imp6,f7,imp7,f8,imp8,f9,imp9,f10,imp10
0,IG,0,A,4,0.033501,151,0.015382,143,0.014524,102,...,113,0.010374,149,0.010283,91,0.008126,56,0.007921,90,0.007541
1,IG,0,B,4,0.036726,143,0.026886,102,0.022346,151,...,149,0.012273,56,0.012219,53,0.011841,150,0.011832,54,0.011456
2,IG,1,A,4,0.039757,151,0.015078,149,0.013708,102,...,53,0.009331,143,0.009183,56,0.008470,54,0.008464,113,0.008107
3,IG,1,B,4,0.026174,149,0.010015,56,0.009178,53,...,150,0.008350,143,0.008005,104,0.007991,54,0.007244,151,0.007053
4,IG,2,A,4,0.018922,102,0.008861,104,0.008651,150,...,149,0.007288,35,0.006228,59,0.004681,56,0.004671,65,0.004670
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63,IG,31,B,104,0.012923,102,0.006588,140,0.005283,54,...,101,0.004657,91,0.004260,4,0.004065,115,0.004024,116,0.003917
64,IG,32,A,101,0.026233,104,0.024202,4,0.022857,102,...,139,0.011622,137,0.009006,116,0.008730,140,0.007116,53,0.006487
65,IG,32,B,140,0.003188,56,0.002994,4,0.002963,53,...,79,0.002659,101,0.002656,102,0.002547,54,0.002513,90,0.002482
66,IG,33,A,4,0.014722,101,0.013880,138,0.013177,104,...,102,0.008431,91,0.007168,115,0.006991,53,0.006137,149,0.005254



================ IG sorted by DriftRatio ================



,pair_id,k_idx,tA,tB,eval_start,eval_end,n_eval_total,n_eval_used,ΔEloc_IG,ΔEglob_IG,σE(A)_IG,σE(B)_IG,DriftRatio_IG,Top10Overlap_IG
0,32,41,42,45,46,48,2029,64,1.056249,0.468798,0.542205,1.143329,1.948061,0.6
1,31,40,41,44,45,47,2779,64,0.999992,0.390965,0.575265,1.011857,1.738315,0.5
2,23,32,33,36,37,39,2437,64,0.812876,0.218657,0.574989,0.648029,1.413725,0.7
3,3,12,13,16,17,19,1945,64,0.659388,0.290642,0.476652,0.587792,1.383372,0.6
4,33,42,43,46,47,49,1793,64,0.936505,0.324454,0.680536,0.968616,1.376127,0.7
5,13,22,23,26,27,29,1664,64,0.814519,0.321187,0.598658,0.594728,1.360574,0.4
6,14,23,24,27,28,30,1982,64,0.848430,0.211933,0.628682,0.658404,1.349538,0.3
7,29,38,39,42,43,45,4182,64,0.888477,0.199327,0.659592,0.602792,1.347011,0.6
8,10,19,20,23,24,26,2237,64,0.734310,0.198256,0.554799,0.576249,1.323561,0.6
9,12,21,22,25,26,28,1007,64,0.660000,0.265367,0.507510,0.574385,1.300469,0.5


In [3]:
# ============================================================
# SAVE RESULTS TO GOOGLE DRIVE (Thesis folder)
# ============================================================

from google.colab import drive
import os
from datetime import datetime


# Path naar Thesis folder
THESIS_DIR = "/content/drive/MyDrive/Thesis"
os.makedirs(THESIS_DIR, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# ----------------------------
# 1. Performance + drift results
# ----------------------------

if "results_df" in globals():
    perf_path = os.path.join(THESIS_DIR, f"performance_drift_results_{timestamp}.csv")
    results_df.to_csv(perf_path, index=False)
    print("Saved:", perf_path)

if "results_ig_df" in globals():
    ig_path = os.path.join(THESIS_DIR, f"IG_drift_results_{timestamp}.csv")
    results_ig_df.to_csv(ig_path, index=False)
    print("Saved:", ig_path)

# ----------------------------
# 2. Explanation details
# ----------------------------

if "top10_ig_df" in globals():
    top10_long_path = os.path.join(THESIS_DIR, f"IG_top10_features_long_{timestamp}.csv")
    top10_ig_df.to_csv(top10_long_path, index=False)
    print("Saved:", top10_long_path)

if "top10_ig_wide_df" in globals():
    top10_wide_path = os.path.join(THESIS_DIR, f"IG_top10_features_wide_{timestamp}.csv")
    top10_ig_wide_df.to_csv(top10_wide_path, index=False)
    print("Saved:", top10_wide_path)

print("\nAll results saved to:", THESIS_DIR)

Saved: /content/drive/MyDrive/Thesis/performance_drift_results_20260309_120930.csv
Saved: /content/drive/MyDrive/Thesis/IG_drift_results_20260309_120930.csv
Saved: /content/drive/MyDrive/Thesis/IG_top10_features_long_20260309_120930.csv
Saved: /content/drive/MyDrive/Thesis/IG_top10_features_wide_20260309_120930.csv

All results saved to: /content/drive/MyDrive/Thesis
